In [1]:
import math
import time

import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from utils.config import (
    SEED,
    MAX_FEATURES,
    MAX_LEN,
    EMBED_DIM,
    NHEAD,
    FFN_DIM,
    NUM_LAYERS,
    DROPOUT,
    BATCH_SIZE,
    EPOCHS,
    LEARNING_RATE,
    NUM_CLASSES,
    device,
    set_seed,
)

from utils.tokenizer import SimpleTokenizer

from utils.metrics import (
    calculate_metrics,
    print_metrics,
    get_classification_report,
)

from utils.training import (
    train_model,
    evaluate,
)


# =========================================================
# Dataset
# =========================================================

class EmotionDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len
    ):

        self.texts = texts.tolist()
        self.labels = labels.tolist()

        self.tokenizer = tokenizer
        self.max_len = max_len

        # 미리 tokenization
        self.encoded_texts = [
            tokenizer.encode(
                text,
                max_length=max_len
            )
            for text in self.texts
        ]


    def __len__(self):

        return len(self.labels)


    def __getitem__(self, idx):

        return {
            "input_ids": self.encoded_texts[idx],
            "labels": self.labels[idx]
        }


# =========================================================
# Collate Function
# =========================================================

def collate_fn(batch):

    input_ids_list = [
        item["input_ids"]
        for item in batch
    ]

    labels = torch.tensor(
        [
            item["labels"]
            for item in batch
        ],
        dtype=torch.long
    )

    # Batch 내 가장 긴 sequence 길이
    max_length = max(
        len(input_ids)
        for input_ids in input_ids_list
    )

    padded_input_ids = []
    attention_masks = []

    for input_ids in input_ids_list:

        padding_length = (
            max_length - len(input_ids)
        )

        # PAD ID = 0
        padded_ids = (
            input_ids
            + [0] * padding_length
        )

        # 실제 token = 1
        # PAD = 0
        attention_mask = (
            [1] * len(input_ids)
            + [0] * padding_length
        )

        padded_input_ids.append(
            padded_ids
        )

        attention_masks.append(
            attention_mask
        )

    input_ids = torch.tensor(
        padded_input_ids,
        dtype=torch.long
    )

    attention_mask = torch.tensor(
        attention_masks,
        dtype=torch.long
    )

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }


# =========================================================
# Positional Encoding
# =========================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        embed_dim,
        max_len
    ):

        super().__init__()

        position = torch.arange(
            max_len,
            dtype=torch.float
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                embed_dim,
                2,
                dtype=torch.float
            )
            * (
                -math.log(10000.0)
                / embed_dim
            )
        )

        pe = torch.zeros(
            max_len,
            embed_dim
        )

        # 짝수 차원
        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        # 홀수 차원
        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        # [1, max_len, embed_dim]
        pe = pe.unsqueeze(0)

        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        seq_len = x.size(1)

        return (
            x
            + self.pe[:, :seq_len]
        )


# =========================================================
# Multi-Head Self-Attention
# =========================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.1
    ):

        super().__init__()

        assert (
            embed_dim % num_heads == 0
        )

        self.embed_dim = embed_dim

        self.num_heads = num_heads

        self.head_dim = (
            embed_dim // num_heads
        )

        # Q Projection
        self.q_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # K Projection
        self.k_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # V Projection
        self.v_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # Output Projection
        self.out_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        attention_mask=None
    ):

        batch_size, seq_len, _ = x.shape

        # -------------------------------------------------
        # Q, K, V
        # -------------------------------------------------

        Q = self.q_proj(x)

        K = self.k_proj(x)

        V = self.v_proj(x)


        # -------------------------------------------------
        # Multi-Head 분할
        # -------------------------------------------------

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)


        # -------------------------------------------------
        # Scaled Dot-Product Attention
        # -------------------------------------------------

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = (
            scores
            / math.sqrt(self.head_dim)
        )


        # -------------------------------------------------
        # Padding Mask
        # -------------------------------------------------

        if attention_mask is not None:

            mask = (
                attention_mask
                .unsqueeze(1)
                .unsqueeze(2)
            )

            scores = scores.masked_fill(
                mask == 0,
                -1e9
            )


        # -------------------------------------------------
        # Softmax
        # -------------------------------------------------

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )


        # -------------------------------------------------
        # Attention × V
        # -------------------------------------------------

        context = torch.matmul(
            attention_weights,
            V
        )


        # -------------------------------------------------
        # Head 합치기
        # -------------------------------------------------

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            self.embed_dim
        )


        # -------------------------------------------------
        # Output Projection
        # -------------------------------------------------

        output = self.out_proj(
            context
        )

        return output


# =========================================================
# Transformer Encoder Layer
# =========================================================

class TransformerEncoderLayerScratch(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):

        super().__init__()


        # -------------------------------------------------
        # Self-Attention
        # -------------------------------------------------

        self.self_attention = (
            MultiHeadSelfAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
        )


        # -------------------------------------------------
        # LayerNorm 1
        # -------------------------------------------------

        self.norm1 = nn.LayerNorm(
            embed_dim
        )


        # -------------------------------------------------
        # Feed Forward Network
        # -------------------------------------------------

        self.ffn = nn.Sequential(

            nn.Linear(
                embed_dim,
                ffn_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                ffn_dim,
                embed_dim
            )
        )


        # -------------------------------------------------
        # LayerNorm 2
        # -------------------------------------------------

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        attention_mask=None
    ):

        # -------------------------------------------------
        # Multi-Head Self-Attention
        # -------------------------------------------------

        attention_output = (
            self.self_attention(
                x,
                attention_mask
            )
        )


        # -------------------------------------------------
        # Add & Norm
        # -------------------------------------------------

        x = self.norm1(
            x
            + self.dropout(
                attention_output
            )
        )


        # -------------------------------------------------
        # Feed Forward
        # -------------------------------------------------

        ffn_output = self.ffn(x)


        # -------------------------------------------------
        # Add & Norm
        # -------------------------------------------------

        x = self.norm2(
            x
            + self.dropout(
                ffn_output
            )
        )

        return x


# =========================================================
# Transformer Encoder
# =========================================================

class TransformerEncoderScratch(
    nn.Module
):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        dropout=0.1
    ):

        super().__init__()

        self.layers = nn.ModuleList(
            [
                TransformerEncoderLayerScratch(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    ffn_dim=ffn_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )


    def forward(
        self,
        x,
        attention_mask=None
    ):

        for layer in self.layers:

            x = layer(
                x,
                attention_mask
            )

        return x


# =========================================================
# Transformer Classifier
# =========================================================

class TransformerClassifier(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        num_classes,
        max_len,
        dropout=0.1
    ):

        super().__init__()


        # -------------------------------------------------
        # Token Embedding
        # -------------------------------------------------

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )


        # -------------------------------------------------
        # Positional Encoding
        # -------------------------------------------------

        self.position_encoding = (
            PositionalEncoding(
                embed_dim=embed_dim,
                max_len=max_len
            )
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # -------------------------------------------------
        # Transformer Encoder
        # -------------------------------------------------

        self.encoder = (
            TransformerEncoderScratch(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                num_layers=num_layers,
                dropout=dropout
            )
        )


        # -------------------------------------------------
        # Classification Head
        # -------------------------------------------------

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )


    def forward(
        self,
        input_ids,
        attention_mask
    ):

        # -------------------------------------------------
        # Embedding
        # -------------------------------------------------

        x = self.embedding(
            input_ids
        )


        # -------------------------------------------------
        # Positional Encoding
        # -------------------------------------------------

        x = self.position_encoding(
            x
        )

        x = self.dropout(x)


        # -------------------------------------------------
        # Transformer Encoder
        # -------------------------------------------------

        x = self.encoder(
            x,
            attention_mask
        )


        # -------------------------------------------------
        # CLS Representation
        # -------------------------------------------------

        cls_output = x[:, 0, :]


        # -------------------------------------------------
        # Classification
        # -------------------------------------------------

        logits = self.classifier(
            cls_output
        )

        return logits


# =========================================================
# Training Curve
# =========================================================

def plot_training_curve(history):

    epochs = range(
        1,
        len(
            history["train_loss"]
        ) + 1
    )

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        epochs,
        history["train_loss"],
        label="Train Loss"
    )

    plt.plot(
        epochs,
        history["val_loss"],
        label="Validation Loss"
    )

    plt.xlabel(
        "Epoch"
    )

    plt.ylabel(
        "Loss"
    )

    plt.title(
        "Transformer Scratch Training Curve"
    )

    plt.legend()

    plt.tight_layout()

    plt.savefig(
        "results/"
        "transformer_scratch_training_curve.png"
    )

    plt.close()


# =========================================================
# Main
# =========================================================

def main():

    # =====================================================
    # Seed
    # =====================================================

    set_seed(SEED)


    print("=" * 70)
    print("05. Transformer from Scratch")
    print("=" * 70)

    print(
        f"Device        : {device}"
    )

    print(
        f"Max Length    : {MAX_LEN}"
    )

    print(
        f"Embedding Dim : {EMBED_DIM}"
    )

    print(
        f"Num Heads     : {NHEAD}"
    )

    print(
        f"FFN Dim       : {FFN_DIM}"
    )

    print(
        f"Num Layers    : {NUM_LAYERS}"
    )

    print(
        f"Dropout       : {DROPOUT}"
    )

    print(
        f"Batch Size    : {BATCH_SIZE}"
    )

    print(
        f"Epochs        : {EPOCHS}"
    )

    print(
        f"Learning Rate : {LEARNING_RATE}"
    )

    print()


    # =====================================================
    # Load Fixed Split
    # =====================================================

    split_path = "results/splits"


    train_df = pd.read_csv(
        f"{split_path}/train.csv"
    )

    val_df = pd.read_csv(
        f"{split_path}/validation.csv"
    )

    test_df = pd.read_csv(
        f"{split_path}/test.csv"
    )


    print("Dataset")
    print("-" * 70)

    print(
        f"Train      : "
        f"{len(train_df):,}"
    )

    print(
        f"Validation : "
        f"{len(val_df):,}"
    )

    print(
        f"Test       : "
        f"{len(test_df):,}"
    )

    print()


    # =====================================================
    # Build Vocabulary
    # =====================================================

    tokenizer = SimpleTokenizer(
        vocab_size=MAX_FEATURES
    )

    # 반드시 train 데이터만 사용
    tokenizer.build_vocab(
        train_df["text"]
    )


    # =====================================================
    # Dataset
    # =====================================================

    print("Encoding Dataset")
    print("-" * 70)

    train_dataset = EmotionDataset(
        texts=train_df["text"],
        labels=train_df["label"],
        tokenizer=tokenizer,
        max_len=MAX_LEN
    )

    val_dataset = EmotionDataset(
        texts=val_df["text"],
        labels=val_df["label"],
        tokenizer=tokenizer,
        max_len=MAX_LEN
    )

    test_dataset = EmotionDataset(
        texts=test_df["text"],
        labels=test_df["label"],
        tokenizer=tokenizer,
        max_len=MAX_LEN
    )

    print("Encoding Complete")
    print()


    # =====================================================
    # DataLoader
    # =====================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )


    # =====================================================
    # Model
    # =====================================================

    vocab_size = len(
        tokenizer.token_to_id
    )


    model = TransformerClassifier(
        vocab_size=vocab_size,
        embed_dim=EMBED_DIM,
        num_heads=NHEAD,
        ffn_dim=FFN_DIM,
        num_layers=NUM_LAYERS,
        num_classes=NUM_CLASSES,
        max_len=MAX_LEN,
        dropout=DROPOUT
    ).to(device)


    # =====================================================
    # Parameter Count
    # =====================================================

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


    print("Model")
    print("-" * 70)

    print(model)

    print()

    print(
        f"Vocabulary Size      : "
        f"{vocab_size:,}"
    )

    print(
        f"Total Parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_params:,}"
    )

    print()


    # =====================================================
    # Loss
    # =====================================================

    criterion = nn.CrossEntropyLoss()


    # =====================================================
    # Optimizer
    # =====================================================

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )


    # =====================================================
    # Training
    # =====================================================

    print("=" * 70)
    print("Training")
    print("=" * 70)


    start_time = time.perf_counter()


    model, history, best_val_macro_f1, _ = (
        train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            criterion=criterion,
            epochs=EPOCHS,
            device=device
        )
    )


    training_time = (
        time.perf_counter()
        - start_time
    )


    print()

    print(
        f"Best Validation Macro F1 : "
        f"{best_val_macro_f1:.4f}"
    )

    print(
        f"Training Time            : "
        f"{training_time:.2f} sec"
    )

    print()


    # =====================================================
    # Test Evaluation
    # =====================================================

    print("=" * 70)
    print("Test Evaluation")
    print("=" * 70)


    test_start_time = (
        time.perf_counter()
    )


    test_loss, y_true, y_pred = evaluate(
        model=model,
        dataloader=test_loader,
        criterion=criterion,
        device=device
    )


    test_inference_time = (
        time.perf_counter()
        - test_start_time
    )


    # =====================================================
    # Metrics
    # =====================================================

    metrics = calculate_metrics(
        y_true,
        y_pred
    )


    print()

    print_metrics(
        metrics
    )

    print(
        f"Test Loss          : "
        f"{test_loss:.4f}"
    )

    print(
        f"Test Inference Time: "
        f"{test_inference_time:.2f} sec"
    )

    print()


    # =====================================================
    # Classification Report
    # =====================================================

    print(
        "Classification Report"
    )

    print("-" * 70)

    print(
        get_classification_report(
            y_true,
            y_pred
        )
    )


    # =====================================================
    # Save Model
    # =====================================================

    model_path = (
        "results/checkpoints/"
        "transformer_scratch.pt"
    )

    torch.save(
        model.state_dict(),
        model_path
    )


    # =====================================================
    # Save Vocabulary
    # =====================================================

    vocab_path = (
        "results/checkpoints/"
        "transformer_scratch_vocab.json"
    )

    tokenizer.save(
        vocab_path
    )


    # =====================================================
    # Training Curve
    # =====================================================

    plot_training_curve(
        history
    )


    # =====================================================
    # Final
    # =====================================================

    print("=" * 70)
    print("Saved Files")
    print("=" * 70)

    print(
        model_path
    )

    print(
        vocab_path
    )

    print(
        "results/"
        "transformer_scratch_training_curve.png"
    )

    print()


# =========================================================
# Run
# =========================================================

if __name__ == "__main__":
    main()

05. Transformer from Scratch
Device        : cuda
Max Length    : 100
Embedding Dim : 128
Num Heads     : 4
FFN Dim       : 256
Num Layers    : 2
Dropout       : 0.1
Batch Size    : 64
Epochs        : 10
Learning Rate : 0.001

Dataset
----------------------------------------------------------------------
Train      : 305,345
Validation : 65,422
Test       : 65,355

Vocabulary
----------------------------------------------------------------------
Vocabulary Size : 1,000

Encoding Dataset
----------------------------------------------------------------------
Encoding Complete

Model
----------------------------------------------------------------------
TransformerClassifier(
  (embedding): Embedding(1000, 128, padding_idx=0)
  (position_encoding): PositionalEncoding()
  (dropout): Dropout(p=0.1, inplace=False)
  (encoder): TransformerEncoderScratch(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayerScratch(
        (self_attention): MultiHeadSelfAttention(
          (q_pr

In [2]:
import math

import pandas as pd

import torch
import torch.nn as nn

from utils.config import (
    SEED,
    MAX_FEATURES,
    MAX_LEN,
    EMBED_DIM,
    NHEAD,
    FFN_DIM,
    NUM_LAYERS,
    DROPOUT,
    device,
    set_seed,
)

from utils.tokenizer import SimpleTokenizer


# =========================================================
# 1. Multi-Head Self-Attention
# =========================================================

class MultiHeadSelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.1
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.head_dim = (
            embed_dim // num_heads
        )

        # Query
        self.q_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # Key
        self.k_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # Value
        self.v_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        # Attention 결과 변환
        self.out_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        padding_mask=None,
        causal_mask=None
    ):

        batch_size, seq_len, _ = x.shape


        # =================================================
        # 1. Q, K, V 생성
        # =================================================

        Q = self.q_proj(x)

        K = self.k_proj(x)

        V = self.v_proj(x)


        # =================================================
        # 2. Multi-Head 분할
        # =================================================

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)


        # =================================================
        # 3. Scaled Dot-Product Attention
        # =================================================

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = (
            scores
            / math.sqrt(self.head_dim)
        )


        # =================================================
        # 4. Padding Mask
        # =================================================

        if padding_mask is not None:

            padding_mask = (
                padding_mask
                .unsqueeze(1)
                .unsqueeze(2)
            )

            scores = scores.masked_fill(
                padding_mask == 0,
                -1e9
            )


        # =================================================
        # 5. Causal Mask
        # =================================================

        if causal_mask is not None:

            scores = scores.masked_fill(
                causal_mask
                .unsqueeze(0)
                .unsqueeze(0),
                -1e9
            )


        # =================================================
        # 6. Softmax
        # =================================================

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )


        # =================================================
        # 7. Attention × Value
        # =================================================

        context = torch.matmul(
            attention_weights,
            V
        )


        # =================================================
        # 8. Multi-Head 결과 결합
        # =================================================

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            self.embed_dim
        )


        # =================================================
        # 9. Output Projection
        # =================================================

        output = self.out_proj(
            context
        )

        return output


# =========================================================
# 2. Multi-Head Cross-Attention
# =========================================================

class MultiHeadCrossAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.1
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.head_dim = (
            embed_dim // num_heads
        )


        # Query
        # Decoder에서 생성
        self.q_proj = nn.Linear(
            embed_dim,
            embed_dim
        )


        # Key
        # Encoder output에서 생성
        self.k_proj = nn.Linear(
            embed_dim,
            embed_dim
        )


        # Value
        # Encoder output에서 생성
        self.v_proj = nn.Linear(
            embed_dim,
            embed_dim
        )


        self.out_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        decoder_x,
        encoder_output,
        encoder_padding_mask=None
    ):

        batch_size = (
            decoder_x.size(0)
        )

        decoder_seq_len = (
            decoder_x.size(1)
        )

        encoder_seq_len = (
            encoder_output.size(1)
        )


        # =================================================
        # 1. Q
        # =================================================

        Q = self.q_proj(
            decoder_x
        )


        # =================================================
        # 2. K, V
        # =================================================

        K = self.k_proj(
            encoder_output
        )

        V = self.v_proj(
            encoder_output
        )


        # =================================================
        # 3. Multi-Head 분할
        # =================================================

        Q = Q.view(
            batch_size,
            decoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            encoder_seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)


        # =================================================
        # 4. Attention Score
        # =================================================

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = (
            scores
            / math.sqrt(self.head_dim)
        )


        # =================================================
        # 5. Encoder Padding Mask
        # =================================================

        if encoder_padding_mask is not None:

            encoder_padding_mask = (
                encoder_padding_mask
                .unsqueeze(1)
                .unsqueeze(2)
            )

            scores = scores.masked_fill(
                encoder_padding_mask == 0,
                -1e9
            )


        # =================================================
        # 6. Softmax
        # =================================================

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )


        # =================================================
        # 7. Attention × Value
        # =================================================

        context = torch.matmul(
            attention_weights,
            V
        )


        # =================================================
        # 8. Multi-Head 결합
        # =================================================

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            decoder_seq_len,
            self.embed_dim
        )


        # =================================================
        # 9. Output Projection
        # =================================================

        output = self.out_proj(
            context
        )

        return output


# =========================================================
# 3. Positional Encoding
# =========================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        embed_dim,
        max_len
    ):
        super().__init__()


        # 위치
        position = torch.arange(
            max_len,
            dtype=torch.float
        ).unsqueeze(1)


        # 주파수
        div_term = torch.exp(
            torch.arange(
                0,
                embed_dim,
                2,
                dtype=torch.float
            )
            * (
                -math.log(10000.0)
                / embed_dim
            )
        )


        # Positional Encoding
        pe = torch.zeros(
            max_len,
            embed_dim
        )


        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )


        # Batch 차원 추가
        pe = pe.unsqueeze(0)


        # 모델 학습 대상이 아닌 buffer
        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        seq_len = x.size(1)

        return (
            x
            + self.pe[:, :seq_len]
        )


# =========================================================
# 4. Transformer Encoder Layer
# =========================================================

class EncoderLayer(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()


        # =================================================
        # Self-Attention
        # =================================================

        self.self_attention = (
            MultiHeadSelfAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
        )


        # =================================================
        # LayerNorm 1
        # =================================================

        self.norm1 = nn.LayerNorm(
            embed_dim
        )


        # =================================================
        # Feed Forward Network
        # =================================================

        self.ffn = nn.Sequential(

            nn.Linear(
                embed_dim,
                ffn_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                ffn_dim,
                embed_dim
            )
        )


        # =================================================
        # LayerNorm 2
        # =================================================

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        padding_mask=None
    ):

        # =================================================
        # 1. Self-Attention
        # =================================================

        attention_output = (
            self.self_attention(
                x,
                padding_mask=padding_mask
            )
        )


        # =================================================
        # 2. Residual + LayerNorm
        # =================================================

        x = self.norm1(
            x
            + self.dropout(
                attention_output
            )
        )


        # =================================================
        # 3. Feed Forward
        # =================================================

        ffn_output = self.ffn(x)


        # =================================================
        # 4. Residual + LayerNorm
        # =================================================

        x = self.norm2(
            x
            + self.dropout(
                ffn_output
            )
        )

        return x


# =========================================================
# 5. Transformer Encoder
# =========================================================

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        dropout=0.1
    ):
        super().__init__()


        self.layers = nn.ModuleList(
            [
                EncoderLayer(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    ffn_dim=ffn_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )


    def forward(
        self,
        x,
        padding_mask=None
    ):

        for layer in self.layers:

            x = layer(
                x,
                padding_mask=padding_mask
            )

        return x


# =========================================================
# 6. Transformer Decoder Layer
# =========================================================

class DecoderLayer(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()


        # =================================================
        # 1. Masked Self-Attention
        # =================================================

        self.self_attention = (
            MultiHeadSelfAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
        )


        self.norm1 = nn.LayerNorm(
            embed_dim
        )


        # =================================================
        # 2. Cross-Attention
        # =================================================

        self.cross_attention = (
            MultiHeadCrossAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
        )


        self.norm2 = nn.LayerNorm(
            embed_dim
        )


        # =================================================
        # 3. Feed Forward
        # =================================================

        self.ffn = nn.Sequential(

            nn.Linear(
                embed_dim,
                ffn_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                ffn_dim,
                embed_dim
            )
        )


        self.norm3 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        encoder_output,
        causal_mask=None,
        encoder_padding_mask=None
    ):

        # =================================================
        # 1. Masked Self-Attention
        # =================================================

        self_attention_output = (
            self.self_attention(
                x,
                causal_mask=causal_mask
            )
        )


        # =================================================
        # Residual + LayerNorm
        # =================================================

        x = self.norm1(
            x
            + self.dropout(
                self_attention_output
            )
        )


        # =================================================
        # 2. Cross-Attention
        # =================================================

        cross_attention_output = (
            self.cross_attention(
                decoder_x=x,
                encoder_output=encoder_output,
                encoder_padding_mask=(
                    encoder_padding_mask
                )
            )
        )


        # =================================================
        # Residual + LayerNorm
        # =================================================

        x = self.norm2(
            x
            + self.dropout(
                cross_attention_output
            )
        )


        # =================================================
        # 3. Feed Forward
        # =================================================

        ffn_output = self.ffn(x)


        # =================================================
        # Residual + LayerNorm
        # =================================================

        x = self.norm3(
            x
            + self.dropout(
                ffn_output
            )
        )

        return x


# =========================================================
# 7. Transformer Decoder
# =========================================================

class TransformerDecoder(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        dropout=0.1
    ):
        super().__init__()


        self.layers = nn.ModuleList(
            [
                DecoderLayer(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    ffn_dim=ffn_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )


    def forward(
        self,
        x,
        encoder_output,
        causal_mask=None,
        encoder_padding_mask=None
    ):

        for layer in self.layers:

            x = layer(
                x=x,
                encoder_output=encoder_output,
                causal_mask=causal_mask,
                encoder_padding_mask=(
                    encoder_padding_mask
                )
            )

        return x


# =========================================================
# 8. Encoder-Decoder Transformer
# =========================================================

class TransformerEncoderDecoder(
    nn.Module
):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        max_len,
        dropout=0.1
    ):
        super().__init__()


        # =================================================
        # Token Embedding
        # =================================================

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )


        # =================================================
        # Positional Encoding
        # =================================================

        self.position_encoding = (
            PositionalEncoding(
                embed_dim=embed_dim,
                max_len=max_len
            )
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # =================================================
        # Encoder
        # =================================================

        self.encoder = (
            TransformerEncoder(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                num_layers=num_layers,
                dropout=dropout
            )
        )


        # =================================================
        # Decoder
        # =================================================

        self.decoder = (
            TransformerDecoder(
                embed_dim=embed_dim,
                num_heads=num_heads,
                ffn_dim=ffn_dim,
                num_layers=num_layers,
                dropout=dropout
            )
        )


    # =====================================================
    # Causal Mask 생성
    # =====================================================

    def create_causal_mask(
        self,
        seq_len,
        device
    ):

        # True  = 차단
        # False = 허용

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

        return causal_mask


    # =====================================================
    # Forward
    # =====================================================

    def forward(
        self,
        encoder_input_ids,
        decoder_input_ids,
        encoder_padding_mask=None
    ):

        # =================================================
        # Encoder
        # =================================================

        encoder_x = self.embedding(
            encoder_input_ids
        )

        encoder_x = self.position_encoding(
            encoder_x
        )

        encoder_x = self.dropout(
            encoder_x
        )


        encoder_output = self.encoder(
            encoder_x,
            padding_mask=(
                encoder_padding_mask
            )
        )


        # =================================================
        # Decoder
        # =================================================

        decoder_x = self.embedding(
            decoder_input_ids
        )

        decoder_x = self.position_encoding(
            decoder_x
        )

        decoder_x = self.dropout(
            decoder_x
        )


        # =================================================
        # Causal Mask
        # =================================================

        decoder_seq_len = (
            decoder_input_ids.size(1)
        )

        causal_mask = (
            self.create_causal_mask(
                seq_len=decoder_seq_len,
                device=(
                    decoder_input_ids.device
                )
            )
        )


        # =================================================
        # Decoder
        # =================================================

        decoder_output = self.decoder(
            x=decoder_x,
            encoder_output=encoder_output,
            causal_mask=causal_mask,
            encoder_padding_mask=(
                encoder_padding_mask
            )
        )


        return (
            encoder_output,
            decoder_output
        )


# =========================================================
# 9. Main
# =========================================================

def main():

    # =====================================================
    # Random Seed
    # =====================================================

    set_seed(SEED)


    print("=" * 70)
    print("05-2. Transformer Encoder-Decoder")
    print("=" * 70)

    print(
        f"Device        : {device}"
    )

    print(
        f"Max Length    : {MAX_LEN}"
    )

    print(
        f"Embedding Dim : {EMBED_DIM}"
    )

    print(
        f"Num Heads     : {NHEAD}"
    )

    print(
        f"FFN Dim       : {FFN_DIM}"
    )

    print(
        f"Num Layers    : {NUM_LAYERS}"
    )

    print(
        f"Dropout       : {DROPOUT}"
    )

    print()


    # =====================================================
    # 1. Load Train Data
    # =====================================================

    train_df = pd.read_csv(
        "results/splits/train.csv"
    )


    print("Dataset")
    print("-" * 70)

    print(
        f"Train Samples : "
        f"{len(train_df):,}"
    )

    print()


    # =====================================================
    # 2. Tokenizer
    # =====================================================

    tokenizer = SimpleTokenizer(
        vocab_size=MAX_FEATURES
    )


    tokenizer.build_vocab(
        train_df["text"]
    )


    # =====================================================
    # 3. Sample Text
    # =====================================================

    sample_text = (
        train_df.iloc[0]["text"]
    )

    sample_label = (
        train_df.iloc[0]["label"]
    )


    # =====================================================
    # 4. Encoder Input
    # =====================================================

    input_ids = tokenizer.encode(
        sample_text,
        max_length=MAX_LEN
    )


    print("Sample")
    print("-" * 70)

    print(
        f"Text  : {sample_text}"
    )

    print(
        f"Label : {sample_label}"
    )

    print(
        f"Token IDs : {input_ids}"
    )

    print()


    # =====================================================
    # 5. Encoder Tensor
    # =====================================================

    encoder_input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )


    # =====================================================
    # 6. Decoder Input
    # =====================================================
    #
    # 실제 생성 모델에서는
    #
    # <START> I am
    #
    # 과 같은 이전 출력 token을
    # Decoder에 입력하게 된다.
    #
    # 여기서는 구조 확인을 위해
    # Encoder input의 일부 token을 사용한다.
    #
    # =====================================================

    decoder_tokens = input_ids[:5]


    decoder_input_ids = torch.tensor(
        [decoder_tokens],
        dtype=torch.long,
        device=device
    )


    # =====================================================
    # 7. Encoder Padding Mask
    # =====================================================

    encoder_padding_mask = (
        encoder_input_ids
        != tokenizer.pad_id
    ).long()


    # =====================================================
    # 8. Input Shape 출력
    # =====================================================

    print("Input Shape")
    print("-" * 70)

    print(
        f"Encoder Input Shape : "
        f"{encoder_input_ids.shape}"
    )

    print(
        f"Decoder Input Shape : "
        f"{decoder_input_ids.shape}"
    )

    print()


    # =====================================================
    # 9. Model 생성
    # =====================================================

    model = TransformerEncoderDecoder(
        vocab_size=len(
            tokenizer.token_to_id
        ),
        embed_dim=EMBED_DIM,
        num_heads=NHEAD,
        ffn_dim=FFN_DIM,
        num_layers=NUM_LAYERS,
        max_len=MAX_LEN,
        dropout=DROPOUT
    ).to(device)


    # =====================================================
    # 10. Parameter 수
    # =====================================================

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )


    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


    print("Model")
    print("-" * 70)

    print(model)

    print()

    print(
        f"Total Parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_params:,}"
    )

    print()


    # =====================================================
    # 11. Forward Pass
    # =====================================================

    model.eval()


    with torch.no_grad():

        (
            encoder_output,
            decoder_output
        ) = model(
            encoder_input_ids=(
                encoder_input_ids
            ),
            decoder_input_ids=(
                decoder_input_ids
            ),
            encoder_padding_mask=(
                encoder_padding_mask
            )
        )


    # =====================================================
    # 12. Output Shape
    # =====================================================

    print("Forward Result")
    print("-" * 70)

    print(
        f"Encoder Output Shape : "
        f"{encoder_output.shape}"
    )

    print(
        f"Decoder Output Shape : "
        f"{decoder_output.shape}"
    )

    print()


    # =====================================================
    # 13. Causal Mask 확인
    # =====================================================

    causal_mask = (
        model.create_causal_mask(
            seq_len=(
                decoder_input_ids.size(1)
            ),
            device=device
        )
    )


    print("Decoder Causal Mask")
    print("-" * 70)

    print(causal_mask)

    print()


    # =====================================================
    # 14. 구조 설명
    # =====================================================

    print("=" * 70)
    print("Architecture Flow")
    print("=" * 70)

    print(
        "Encoder:"
    )

    print(
        "Input"
        " -> Embedding"
        " -> Positional Encoding"
        " -> Self-Attention"
        " -> FFN"
        " -> Encoder Output"
    )

    print()

    print(
        "Decoder:"
    )

    print(
        "Decoder Input"
        " -> Masked Self-Attention"
        " -> Cross-Attention"
        " -> FFN"
        " -> Decoder Output"
    )

    print()

    print(
        "Cross-Attention:"
    )

    print(
        "Query = Decoder"
        " / Key = Encoder"
        " / Value = Encoder"
    )

    print()


    # =====================================================
    # 15. 완료
    # =====================================================

    print("=" * 70)

    print(
        "Encoder-Decoder forward pass "
        "completed successfully."
    )

    print("=" * 70)


# =========================================================
# Entry Point
# =========================================================

if __name__ == "__main__":

    main()

05-2. Transformer Encoder-Decoder
Device        : cuda
Max Length    : 100
Embedding Dim : 128
Num Heads     : 4
FFN Dim       : 256
Num Layers    : 2
Dropout       : 0.1

Dataset
----------------------------------------------------------------------
Train Samples : 305,345

Vocabulary
----------------------------------------------------------------------
Vocabulary Size : 1,000

Sample
----------------------------------------------------------------------
Text  : im feeling rather rotten so im not very ambitious right now
Label : 0
Token IDs : [2, 19, 10, 201, 710, 17, 19, 28, 49, 1, 113, 61, 3]

Input Shape
----------------------------------------------------------------------
Encoder Input Shape : torch.Size([1, 13])
Decoder Input Shape : torch.Size([1, 5])

Model
----------------------------------------------------------------------
TransformerEncoderDecoder(
  (embedding): Embedding(1000, 128, padding_idx=0)
  (position_encoding): PositionalEncoding()
  (dropout): Dropout(p=0.1, i

In [3]:
import math

import pandas as pd

import torch
import torch.nn as nn

from utils.config import (
    SEED,
    MAX_FEATURES,
    MAX_LEN,
    EMBED_DIM,
    NHEAD,
    FFN_DIM,
    NUM_LAYERS,
    DROPOUT,
    NUM_CLASSES,
    device,
    set_seed,
)

from utils.tokenizer import SimpleTokenizer


# =========================================================
# 1. Multi-Head Causal Self-Attention
# =========================================================

class MultiHeadCausalSelfAttention(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.1
    ):
        super().__init__()

        assert embed_dim % num_heads == 0

        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.head_dim = (
            embed_dim // num_heads
        )


        # -------------------------------------------------
        # Query / Key / Value Projection
        # -------------------------------------------------

        self.q_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.k_proj = nn.Linear(
            embed_dim,
            embed_dim
        )

        self.v_proj = nn.Linear(
            embed_dim,
            embed_dim
        )


        # -------------------------------------------------
        # Output Projection
        # -------------------------------------------------

        self.out_proj = nn.Linear(
            embed_dim,
            embed_dim
        )


        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        padding_mask=None,
        causal_mask=None
    ):

        batch_size, seq_len, _ = x.shape


        # =================================================
        # 1. Q, K, V 생성
        # =================================================

        Q = self.q_proj(x)

        K = self.k_proj(x)

        V = self.v_proj(x)


        # =================================================
        # 2. Multi-Head 분할
        # =================================================

        Q = Q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        K = K.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        V = V.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)


        # =================================================
        # 3. Scaled Dot-Product Attention
        # =================================================

        scores = torch.matmul(
            Q,
            K.transpose(-2, -1)
        )

        scores = (
            scores
            / math.sqrt(self.head_dim)
        )


        # =================================================
        # 4. Padding Mask
        # =================================================

        if padding_mask is not None:

            padding_mask = (
                padding_mask
                .unsqueeze(1)
                .unsqueeze(2)
            )

            scores = scores.masked_fill(
                padding_mask == 0,
                -1e9
            )


        # =================================================
        # 5. Causal Mask
        # =================================================

        if causal_mask is not None:

            scores = scores.masked_fill(
                causal_mask
                .unsqueeze(0)
                .unsqueeze(0),
                -1e9
            )


        # =================================================
        # 6. Softmax
        # =================================================

        attention_weights = torch.softmax(
            scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )


        # =================================================
        # 7. Attention × V
        # =================================================

        context = torch.matmul(
            attention_weights,
            V
        )


        # =================================================
        # 8. Multi-Head 결합
        # =================================================

        context = context.transpose(
            1,
            2
        ).contiguous()

        context = context.view(
            batch_size,
            seq_len,
            self.embed_dim
        )


        # =================================================
        # 9. Output Projection
        # =================================================

        output = self.out_proj(
            context
        )

        return output


# =========================================================
# 2. Positional Encoding
# =========================================================

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        embed_dim,
        max_len
    ):
        super().__init__()


        # -------------------------------------------------
        # Position
        # -------------------------------------------------

        position = torch.arange(
            max_len,
            dtype=torch.float
        ).unsqueeze(1)


        # -------------------------------------------------
        # Frequency
        # -------------------------------------------------

        div_term = torch.exp(
            torch.arange(
                0,
                embed_dim,
                2,
                dtype=torch.float
            )
            * (
                -math.log(10000.0)
                / embed_dim
            )
        )


        # -------------------------------------------------
        # Positional Encoding Matrix
        # -------------------------------------------------

        pe = torch.zeros(
            max_len,
            embed_dim
        )


        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )


        # Batch dimension
        pe = pe.unsqueeze(0)


        # 모델 파라미터가 아닌 buffer
        self.register_buffer(
            "pe",
            pe
        )


    def forward(self, x):

        seq_len = x.size(1)

        return (
            x
            + self.pe[:, :seq_len]
        )


# =========================================================
# 3. Decoder Block
# =========================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        embed_dim,
        num_heads,
        ffn_dim,
        dropout=0.1
    ):
        super().__init__()


        # =================================================
        # Causal Self-Attention
        # =================================================

        self.self_attention = (
            MultiHeadCausalSelfAttention(
                embed_dim=embed_dim,
                num_heads=num_heads,
                dropout=dropout
            )
        )


        # =================================================
        # LayerNorm 1
        # =================================================

        self.norm1 = nn.LayerNorm(
            embed_dim
        )


        # =================================================
        # Feed Forward Network
        # =================================================

        self.ffn = nn.Sequential(

            nn.Linear(
                embed_dim,
                ffn_dim
            ),

            nn.ReLU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                ffn_dim,
                embed_dim
            )
        )


        # =================================================
        # LayerNorm 2
        # =================================================

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )


    def forward(
        self,
        x,
        padding_mask=None,
        causal_mask=None
    ):

        # =================================================
        # 1. Causal Self-Attention
        # =================================================

        attention_output = (
            self.self_attention(
                x,
                padding_mask=padding_mask,
                causal_mask=causal_mask
            )
        )


        # =================================================
        # 2. Residual + LayerNorm
        # =================================================

        x = self.norm1(
            x
            + self.dropout(
                attention_output
            )
        )


        # =================================================
        # 3. Feed Forward
        # =================================================

        ffn_output = self.ffn(x)


        # =================================================
        # 4. Residual + LayerNorm
        # =================================================

        x = self.norm2(
            x
            + self.dropout(
                ffn_output
            )
        )

        return x


# =========================================================
# 4. Decoder-only Transformer
# =========================================================

class DecoderOnlyTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim,
        num_heads,
        ffn_dim,
        num_layers,
        max_len,
        num_classes,
        dropout=0.1
    ):
        super().__init__()


        # =================================================
        # Token Embedding
        # =================================================

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )


        # =================================================
        # Positional Encoding
        # =================================================

        self.position_encoding = (
            PositionalEncoding(
                embed_dim=embed_dim,
                max_len=max_len
            )
        )


        self.dropout = nn.Dropout(
            dropout
        )


        # =================================================
        # Decoder Blocks
        # =================================================

        self.layers = nn.ModuleList(
            [
                DecoderBlock(
                    embed_dim=embed_dim,
                    num_heads=num_heads,
                    ffn_dim=ffn_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )


        # =================================================
        # Classification Head
        # =================================================
        #
        # 구조 학습용이지만,
        # Decoder-only representation을 이용해
        # 분류가 어떻게 연결될 수 있는지 확인하기 위해
        # classification head를 포함한다.
        #
        # 이번 단계에서는 학습하지 않는다.
        #
        # =================================================

        self.classifier = nn.Linear(
            embed_dim,
            num_classes
        )


    # =====================================================
    # Causal Mask 생성
    # =====================================================

    def create_causal_mask(
        self,
        seq_len,
        device
    ):

        # True  = 차단
        # False = 허용

        causal_mask = torch.triu(
            torch.ones(
                seq_len,
                seq_len,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

        return causal_mask


    # =====================================================
    # Forward
    # =====================================================

    def forward(
        self,
        input_ids,
        padding_mask=None
    ):

        # =================================================
        # 1. Token Embedding
        # =================================================

        x = self.embedding(
            input_ids
        )


        # =================================================
        # 2. Positional Encoding
        # =================================================

        x = self.position_encoding(
            x
        )


        # =================================================
        # 3. Dropout
        # =================================================

        x = self.dropout(x)


        # =================================================
        # 4. Causal Mask
        # =================================================

        seq_len = input_ids.size(1)

        causal_mask = (
            self.create_causal_mask(
                seq_len=seq_len,
                device=input_ids.device
            )
        )


        # =================================================
        # 5. Decoder Blocks
        # =================================================

        for layer in self.layers:

            x = layer(
                x,
                padding_mask=padding_mask,
                causal_mask=causal_mask
            )


        # =================================================
        # 6. Classification Representation
        # =================================================
        #
        # Decoder-only에서는 [CLS]가 전체 문장을
        # 볼 수 없기 때문에 마지막 유효 token을 사용한다.
        #
        # 예:
        #
        # A B C D
        #
        # D는 A, B, C, D를 모두 볼 수 있다.
        #
        # =================================================

        if padding_mask is not None:

            lengths = (
                padding_mask.sum(dim=1)
            )

            last_token_indices = (
                lengths - 1
            )

            batch_indices = torch.arange(
                x.size(0),
                device=x.device
            )

            pooled_output = x[
                batch_indices,
                last_token_indices
            ]

        else:

            pooled_output = x[:, -1, :]


        # =================================================
        # 7. Classification Head
        # =================================================

        logits = self.classifier(
            pooled_output
        )


        return (
            x,
            pooled_output,
            logits,
            causal_mask
        )


# =========================================================
# 5. Main
# =========================================================

def main():

    # =====================================================
    # Random Seed
    # =====================================================

    set_seed(SEED)


    print("=" * 70)
    print("05-3. Transformer Decoder-only")
    print("=" * 70)


    print(
        f"Device        : {device}"
    )

    print(
        f"Max Length    : {MAX_LEN}"
    )

    print(
        f"Embedding Dim : {EMBED_DIM}"
    )

    print(
        f"Num Heads     : {NHEAD}"
    )

    print(
        f"FFN Dim       : {FFN_DIM}"
    )

    print(
        f"Num Layers    : {NUM_LAYERS}"
    )

    print(
        f"Dropout       : {DROPOUT}"
    )

    print()


    # =====================================================
    # 1. Load Train Data
    # =====================================================

    train_df = pd.read_csv(
        "results/splits/train.csv"
    )


    print("Dataset")
    print("-" * 70)

    print(
        f"Train Samples : "
        f"{len(train_df):,}"
    )

    print()


    # =====================================================
    # 2. Build Vocabulary
    # =====================================================

    tokenizer = SimpleTokenizer(
        vocab_size=MAX_FEATURES
    )


    tokenizer.build_vocab(
        train_df["text"]
    )


    # =====================================================
    # 3. Sample Text
    # =====================================================

    sample_text = (
        train_df.iloc[0]["text"]
    )

    sample_label = (
        train_df.iloc[0]["label"]
    )


    # =====================================================
    # 4. Tokenize
    # =====================================================

    input_ids = tokenizer.encode(
        sample_text,
        max_length=MAX_LEN
    )


    print("Sample")
    print("-" * 70)

    print(
        f"Text      : {sample_text}"
    )

    print(
        f"Label     : {sample_label}"
    )

    print(
        f"Token IDs : {input_ids}"
    )

    print()


    # =====================================================
    # 5. Tensor 생성
    # =====================================================

    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=device
    )


    # =====================================================
    # 6. Padding Mask
    # =====================================================

    padding_mask = (
        input_ids != tokenizer.pad_id
    ).long()


    print("Input Shape")
    print("-" * 70)

    print(
        f"Input Shape : "
        f"{input_ids.shape}"
    )

    print()


    # =====================================================
    # 7. Model 생성
    # =====================================================

    model = DecoderOnlyTransformer(
        vocab_size=len(
            tokenizer.token_to_id
        ),
        embed_dim=EMBED_DIM,
        num_heads=NHEAD,
        ffn_dim=FFN_DIM,
        num_layers=NUM_LAYERS,
        max_len=MAX_LEN,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT
    ).to(device)


    # =====================================================
    # 8. Parameter 수
    # =====================================================

    total_params = sum(
        p.numel()
        for p in model.parameters()
    )


    trainable_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


    print("Model")
    print("-" * 70)

    print(model)

    print()

    print(
        f"Total Parameters     : "
        f"{total_params:,}"
    )

    print(
        f"Trainable Parameters : "
        f"{trainable_params:,}"
    )

    print()


    # =====================================================
    # 9. Forward Pass
    # =====================================================

    model.eval()


    with torch.no_grad():

        (
            hidden_states,
            pooled_output,
            logits,
            causal_mask
        ) = model(
            input_ids=input_ids,
            padding_mask=padding_mask
        )


    # =====================================================
    # 10. Output Shape
    # =====================================================

    print("Forward Result")
    print("-" * 70)

    print(
        f"Hidden States Shape : "
        f"{hidden_states.shape}"
    )

    print(
        f"Pooled Output Shape : "
        f"{pooled_output.shape}"
    )

    print(
        f"Logits Shape        : "
        f"{logits.shape}"
    )

    print()


    # =====================================================
    # 11. Prediction 확인
    # =====================================================

    predicted_class = (
        torch.argmax(
            logits,
            dim=1
        )
        .item()
    )


    print("Classification Output")
    print("-" * 70)

    print(
        f"Target Label     : "
        f"{sample_label}"
    )

    print(
        f"Predicted Class  : "
        f"{predicted_class}"
    )

    print()


    # =====================================================
    # 12. Causal Mask 확인
    # =====================================================

    print("Decoder Causal Mask")
    print("-" * 70)

    print(causal_mask)

    print()


    # =====================================================
    # 13. Architecture Flow
    # =====================================================

    print("=" * 70)
    print("Architecture Flow")
    print("=" * 70)

    print(
        "Input"
        " -> Token Embedding"
        " -> Positional Encoding"
    )

    print(
        " -> Causal Self-Attention"
        " -> Add & Norm"
    )

    print(
        " -> Feed Forward"
        " -> Add & Norm"
    )

    print(
        " -> Decoder Block × "
        f"{NUM_LAYERS}"
    )

    print(
        " -> Last Valid Token"
        " -> Classification Head"
    )

    print()


    # =====================================================
    # 14. Attention 특징
    # =====================================================

    print(
        "Attention:"
    )

    print(
        "Query = Current Token"
        " / Key = Previous + Current Tokens"
        " / Value = Previous + Current Tokens"
    )

    print()


    # =====================================================
    # 15. 완료
    # =====================================================

    print("=" * 70)

    print(
        "Decoder-only forward pass "
        "completed successfully."
    )

    print("=" * 70)


# =========================================================
# Entry Point
# =========================================================

if __name__ == "__main__":

    main()

05-3. Transformer Decoder-only
Device        : cuda
Max Length    : 100
Embedding Dim : 128
Num Heads     : 4
FFN Dim       : 256
Num Layers    : 2
Dropout       : 0.1

Dataset
----------------------------------------------------------------------
Train Samples : 305,345

Vocabulary
----------------------------------------------------------------------
Vocabulary Size : 1,000

Sample
----------------------------------------------------------------------
Text      : im feeling rather rotten so im not very ambitious right now
Label     : 0
Token IDs : [2, 19, 10, 201, 710, 17, 19, 28, 49, 1, 113, 61, 3]

Input Shape
----------------------------------------------------------------------
Input Shape : torch.Size([1, 13])

Model
----------------------------------------------------------------------
DecoderOnlyTransformer(
  (embedding): Embedding(1000, 128, padding_idx=0)
  (position_encoding): PositionalEncoding()
  (dropout): Dropout(p=0.1, inplace=False)
  (layers): ModuleList(
    (0-1)